<a href="https://colab.research.google.com/github/TheLimaLeo/atv1am/blob/main/atv1_am.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Configuração de impressão do NumPy para melhor legibilidade
np.set_printoptions(precision=4, suppress=True)

In [ ]:
# DESAFIO 1: DETECÇÃO DE FRAUDE COM PERCEPTRON


# 1. Base Histórica de Treinamento
# X: [Valor Normalizado (0.0 a 10.0), Frequência de Operações]
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

# Rótulos das classes (0: Legítima, 1: Suspeita)
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])


def treinar_perceptron(X, y, lr=0.1, epocas=50):
    """
    Treina o Perceptron clássico (Rosenblatt) com inicialização de pesos e viés em 1.0.
    Utiliza função degrau: y = 1 se z >= 0 senão 0.
    """
    n_amostras, n_features = X.shape
    w = np.ones(n_features, dtype=np.float64)
    b = 1.0

    for epoca in range(epocas):
        erros_epoca = 0
        for i in range(n_amostras):
            # Combinação linear: z = w · x + b
            z = np.dot(X[i], w) + b
            y_pred = 1 if z >= 0.0 else 0

            # Erro de predição
            erro = y[i] - y_pred

            # Regra de atualização de Rosenblatt quando há divergência
            if erro != 0:
                w += lr * erro * X[i]
                b += lr * erro
                erros_epoca += 1

        # Critério de parada antecipada se convergiu perfeitamente
        if erros_epoca == 0:
            break

    return w, b


def predizer_perceptron(x, w, b):
    """Realiza a inferência para uma amostra x usando w e b calibrados."""
    z = np.dot(x, w) + b
    return 1 if z >= 0.0 else 0


# Treinamento do modelo
taxa_aprendizado = 0.1
max_epocas = 100
w_calibrado, b_calibrado = treinar_perceptron(
    X_treino_fraude,
    y_treino_fraude,
    lr=taxa_aprendizado,
    epocas=max_epocas
)

print(f"Parâmetros Finais Calibrados:")
print(f"Pesos (w): {w_calibrado}")
print(f"Viés (b):  {b_calibrado:.4f}\n")

# Avaliação das Amostras de Teste
amostras_fraude = {
    "Transação A": np.array([5.0, 4.0]),
    "Transação B": np.array([7.0, 6.5])
}

mapa_classes_fraude = {0: "Legítima", 1: "Suspeita (Risco de Fraude)"}

print("Resultados da Inferência em Tempo Real:")
for nome, x_teste in amostras_fraude.items():
    pred = predizer_perceptron(x_teste, w_calibrado, b_calibrado)
    print(f"- {nome} {x_teste.tolist()}: Classe {pred} -> {mapa_classes_fraude[pred]}")

Parâmetros Finais Calibrados:
Pesos (w): [-0.   0.4]
Viés (b):  -1.2000

Resultados da Inferência em Tempo Real:
- Transação A [5.0, 4.0]: Classe 1 -> Suspeita (Risco de Fraude)
- Transação B [7.0, 6.5]: Classe 1 -> Suspeita (Risco de Fraude)


In [ ]:
# DESAFIO 2: RISCO DE CHURN COM K-NEAREST NEIGHBORS (VETORIZADO)


# 1. Base Histórica de Treinamento
# X: [Dias de Inatividade, Chamados Críticos]
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

# Rótulos (0: Baixo Risco, 1: Alto Risco)
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])


def classificar_knn(X_train, y_train, ponto_consulta, k=3, metrica="euclidiana"):
    """
    Classificador KNN vetorizado com suporte a distâncias Euclidiana e Manhattan.
    Determina a classe vencedora por contagem majoritária de votos.
    """
    # Cálculo das distâncias por broadcast vetorizado do NumPy
    if metrica.lower() == "euclidiana":
        distancias = np.sqrt(np.sum((X_train - ponto_consulta) ** 2, axis=1))
    elif metrica.lower() == "manhattan":
        distancias = np.sum(np.abs(X_train - ponto_consulta), axis=1)
    else:
        raise ValueError("Métrica não suportada. Escolha 'euclidiana' ou 'manhattan'.")

    # Identificação dos k vizinhos mais próximos
    indices_ordenados = np.argsort(distancias)
    vizinhos_idx = indices_ordenados[:k]
    vizinhos_dist = distancias[vizinhos_idx]
    vizinhos_rotulos = y_train[vizinhos_idx]

    # Votação majoritária
    contagem_votos = np.bincount(vizinhos_rotulos)
    classe_predita = int(np.argmax(contagem_votos))

    return classe_predita, vizinhos_idx, vizinhos_dist


# Casos de Teste
testes_churn = [
    {"nome": "Cliente 1", "ponto": np.array([13.0, 10.0]), "k": 3, "metrica": "euclidiana"},
    {"nome": "Cliente 2", "ponto": np.array([7.0, 1.5]),   "k": 3, "metrica": "euclidiana"}
]

rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}

print("Diagnóstico de Churn com KNN:")
for teste in testes_churn:
    classe, viz_idx, viz_dist = classificar_knn(
        X_treino_churn,
        y_treino_churn,
        teste["ponto"],
        k=teste["k"],
        metrica=teste["metrica"]
    )

    print(f"\n--- {teste['nome']} {teste['ponto'].tolist()} (k={teste['k']}, Métrica={teste['metrica']}) ---")
    print(f"Classe Predita:     {classe} ({rotulos_churn[classe]})")
    print(f"Índices Vizinhos:   {viz_idx.tolist()}")
    print(f"Distâncias:         {np.round(viz_dist, 4).tolist()}")

Diagnóstico de Churn com KNN:

--- Cliente 1 [13.0, 10.0] (k=3, Métrica=euclidiana) ---
Classe Predita:     1 (Alto Risco)
Índices Vizinhos:   [4, 5, 6]
Distâncias:         [6.3246, 7.0711, 9.2195]

--- Cliente 2 [7.0, 1.5] (k=3, Métrica=euclidiana) ---
Classe Predita:     0 (Baixo Risco)
Índices Vizinhos:   [3, 2, 1]
Distâncias:         [1.118, 2.0616, 4.272]


In [ ]:
# DESAFIO 3: RECOMENDAÇÃO DE HARDWARE POR SIMILARIDADE ESPACIAL


# Catálogo: [vCPUs, Memória RAM (GB), Armazenamento SSD (GB)]
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

# Perfil demandado: [12.0 vCPUs, 28.0 GB RAM, 850.0 GB SSD]
demanda_aplicacao = np.array([12.0, 28.0, 850.0])
top_n = 2


def recomendar_servidores(catalogo, nomes, demanda, k=2):
    """
    Recomenda as instâncias mais próximas no espaço 3D (vCPU, RAM, SSD)
    utilizando distância euclidiana vetorizada.
    """
    # Cálculo vetorizado da distância euclidiana tridimensional
    distancias = np.sqrt(np.sum((catalogo - demanda) ** 2, axis=1))

    # Ordenação das melhores opções
    indices_ordenados = np.argsort(distancias)[:k]

    recomendacoes = []
    for rank, idx in enumerate(indices_ordenados, start=1):
        recomendacoes.append({
            "ranking": rank,
            "nome": nomes[idx],
            "specs": catalogo[idx],
            "distancia": distancias[idx]
        })

    return recomendacoes


# Execução do sizing
ranking_recomendado = recomendar_servidores(
    catalogo_servidores,
    nomes_servidores,
    demanda_aplicacao,
    k=top_n
)

print(f"Demanda do Cliente: {demanda_aplicacao[0]:.0f} vCPUs, {demanda_aplicacao[1]:.0f} GB RAM, {demanda_aplicacao[2]:.0f} GB SSD\n")
print(f"{'Ranking':<10} | {'Servidor':<30} | {'vCPUs':<7} | {'RAM (GB)':<10} | {'SSD (GB)':<10} | {'Distância Euclidiana'}")
print("-" * 95)

for item in ranking_recomendado:
    specs = item["specs"]
    print(f"{str(item['ranking']) + 'º lugar':<10} | "
          f"{item['nome']:<30} | "
          f"{specs[0]:<7.0f} | "
          f"{specs[1]:<10.0f} | "
          f"{specs[2]:<10.0f} | "
          f"{item['distancia']:.4f}")

Demanda do Cliente: 12 vCPUs, 28 GB RAM, 850 GB SSD

Ranking    | Servidor                       | vCPUs   | RAM (GB)   | SSD (GB)   | Distância Euclidiana
-----------------------------------------------------------------------------------------------
1º lugar   | High Performance Computing     | 32      | 64         | 1000       | 155.5506
2º lugar   | Database Enterprise            | 16      | 32         | 500        | 350.0457
